# Tareas

In [ ]:
%pip -q install numpy pandas matplotlib requests pypdf faiss-cpu qdrant-client langchain-core langchain-text-splitters langchain-qdrant langchain-nvidia-ai-endpoints sentence-transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 1.0 MB/s eta 0:00:00


In [4]:
import os, json, time, math, re, csv, textwrap, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
Path('data').mkdir(exist_ok=True); Path('eval').mkdir(exist_ok=True); Path('evidencias').mkdir(exist_ok=True)
RESPUESTAS = ['# RESPUESTAS · SI3003 Workshop RAG', '', 'Resultados generados al ejecutar el notebook. Revisa y complementa con tu interpretación personal.', '']
def section(title, body):
    RESPUESTAS.extend([f'## {title}', '', str(body), ''])
def table_md(df): return df.to_markdown(index=False)
# T1
M=16; N=1_000_000; rng=np.random.default_rng(0)
levels=np.floor(-np.log(1-rng.random(N))/np.log(M)).astype(int)
t1=pd.DataFrame([{'Capa':f'>= {k}','Teórico':N/M**k,'Simulado':int((levels>=k).sum())} for k in [1,2,3]])
display(t1)
section('T1 · Capas HNSW', table_md(t1)+'\n\nEn las capas altas hay menos nodos, lo que permite dar saltos amplios sin explorar una gran cantidad de candidatos. Después se refina la búsqueda en las capas inferiores.')
section('T2 · ef y k', 'Con k=10 y ef=5 no hay capacidad para mantener diez candidatos: ef debe ser como mínimo k. Una elección inicial razonable es ef=max(10, k) o algo mayor según el recall requerido; se aumenta si falta calidad de recuperación, a costa de latencia. ef se puede ajustar en consultas sin reconstruir el índice; ef_construction se usa cuando se construye el grafo, por lo que modificarlo requiere reconstrucción para cambiar las conexiones ya creadas.')
t3=pd.DataFrame([
 {'Aspecto':'¿Se sortea capa?','Inserción':'Sí, asigna nivel al nodo nuevo','Búsqueda':'No, recorre capas existentes','Por qué':'Solo la inserción crea nodos y niveles'},
 {'Aspecto':'ef en capas altas','Inserción':'Normalmente 1 para navegación inicial','Búsqueda':'Normalmente 1 para navegación inicial','Por qué':'En la parte alta interesa acercarse rápidamente'},
 {'Aspecto':'Fase principal','Inserción':'efConstruction, conexiones y vecinos','Búsqueda':'efSearch (ef), k resultados','Por qué':'Una construye el grafo; otra consulta'},
 {'Aspecto':'¿Modifica el índice?','Inserción':'Sí','Búsqueda':'No','Por qué':'Consultar no añade vectores'},
])
section('T3 · Inserción y búsqueda',table_md(t3)+'\n\n`vectorstore.add_documents(...)` inserta y `retriever.invoke(...)` busca.')
t4=pd.DataFrame({'Posición FAISS':[0,1,2,3], 'Antes':['A','B','C','D'], 'Después de borrar B sin actualizar FAISS':['A','C','D','IndexError']})
section('T4 · Invariante FAISS',table_md(t4)+'\n\nLa búsqueda original [1,2,3] pasa a devolver [C,D,IndexError] al consultar la lista. Las posiciones 1 y 2 generan errores silenciosos de correspondencia; la posición 3 produce una excepción por índice fuera de rango. Un store vectorial administrado integra IDs y payloads para mantener la correspondencia entre vector y documento (sin eliminar la necesidad de manejar borrados correctamente).')
section('T5 · Fuerza bruta', '768 000 000 multiplicaciones/consulta × 100 consultas/s = 76 800 000 000 multiplicaciones/s = 7.68 × 10^10 multiplicaciones/s, sin incluir otras operaciones o costos de memoria. La búsqueda exacta puede volverse demasiado costosa en grandes colecciones y altos QPS. ANN sacrifica exactitud/recall para ganar velocidad y escalabilidad.')
print('✅ Teoría calculada')

,Capa,Teórico,Simulado
0,>= 1,62500.000000,62675
1,>= 2,3906.250000,4005
2,>= 3,244.140625,270


✅ Teoría calculada


In [6]:
from getpass import getpass
try:
    from google.colab import userdata
    API_KEY=userdata.get('NVIDIA_API_KEY')
except Exception: API_KEY=None
if not API_KEY:
    API_KEY=getpass('NVIDIA_API_KEY (Enter sin clave = embeddings locales / respuestas extractivas): ').strip()
if API_KEY: os.environ['NVIDIA_API_KEY']=API_KEY
GENERATION_MODEL='nvidia/nemotron-3-super-120b-a12b'
EMBEDDING_MODEL='nvidia/nemotron-3-embed-1b'
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
llm=None
if API_KEY:
    try:
        llm=ChatNVIDIA(model=GENERATION_MODEL,temperature=0.1,max_tokens=450)
        _=llm.invoke('Responde solamente OK.')
        print('✅ LLM NVIDIA funcional:', GENERATION_MODEL)
    except Exception as exc:
        llm=None; print('⚠️ NVIDIA generación no disponible:',str(exc)[:240])
embeddings=None
if API_KEY:
    try:
        embeddings=NVIDIAEmbeddings(model=EMBEDDING_MODEL,truncate='END')
        EMBEDDING_DIM=len(embeddings.embed_query('prueba'))
        print('✅ NVIDIA embeddings:',EMBEDDING_DIM)
    except Exception as exc:
        print('⚠️ NVIDIA embeddings no disponibles:',str(exc)[:240])
if embeddings is None:
    from langchain_core.embeddings import Embeddings
    from sentence_transformers import SentenceTransformer
    class LocalEmbeddings(Embeddings):
        def __init__(self): self.model=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
        def embed_documents(self,texts):return self.model.encode(list(texts),normalize_embeddings=True,show_progress_bar=False).tolist()
        def embed_query(self,text):return self.model.encode([text],normalize_embeddings=True)[0].tolist()
    embeddings=LocalEmbeddings(); EMBEDDING_DIM=len(embeddings.embed_query('prueba'))
    print('✅ Embeddings locales multilingües:', EMBEDDING_DIM)
if llm is None: print('⚠️ Respuestas extractivas (sin generación NVIDIA): no son respuestas de un LLM. Quedará documentado en el reporte.')

/tmp/ipykernel_6744/3854883498.py:21: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm=ChatNVIDIA(model=GENERATION_MODEL,temperature=0.1,max_tokens=450)


⚠️ NVIDIA generación no disponible: [404] Not Found
Function 'ac74040f-9fc9-4c5e-ac74-279ba5161d69': Not found for account 'NqsLUkxhPwjV-eTrbVPKu-6gwwwkTNBglG9bpr2ste0'


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ NVIDIA embeddings: 2048
⚠️ Respuestas extractivas (sin generación NVIDIA): no son respuestas de un LLM. Quedará documentado en el reporte.


In [7]:
DEMO=[
 {'id':'demo1','title':'CloudSync Pro — Precios','content':'El plan Enterprise de CloudSync Pro cuesta $49/mes por usuario con compromiso anual.','metadata':{'department':'ventas','type':'pricing'}},
 {'id':'demo2','title':'Política de trabajo remoto','content':'Los empleados pueden trabajar remotamente hasta 3 días por semana, con aprobación del jefe directo.','metadata':{'department':'recursos humanos','type':'policy'}},
 {'id':'demo3','title':'Política de reembolsos','content':'Los reembolsos mayores de $100 requieren aprobación del supervisor; se procesan en 3 a 5 días hábiles.','metadata':{'department':'finanzas','type':'policy'}},
 {'id':'demo4','title':'Onboarding','content':'Durante la primera semana de onboarding se completan módulos obligatorios de seguridad, cumplimiento y cultura organizacional.','metadata':{'department':'recursos humanos','type':'guide'}}]
OWN=[
 ('sw001','DirectorioSW — Propósito','DirectorioSW es una comunidad y directorio de emprendimientos liderados por mujeres. El objetivo es facilitar recomendaciones de confianza y dar visibilidad a negocios seleccionados.','comunidad','descripcion'),
 ('sw002','DirectorioSW — Postulación','La postulación al programa de Aliadas Fundadoras tiene un costo único de 10000 pesos colombianos. Toda solicitud se revisa manualmente antes de ser aprobada.','operaciones','politica'),
 ('sw003','DirectorioSW — Cupos','La convocatoria inicial ofrece 50 cupos de Aliadas Fundadoras. Las postulaciones se consideran en el orden de recepción, sujeto a validación de requisitos.','operaciones','convocatoria'),
 ('sw004','DirectorioSW — Requisitos','Para ser Aliada Fundadora, el negocio debe ser liderado por una mujer y aceptar las reglas de publicación de la comunidad.','operaciones','politica'),
 ('sw005','DirectorioSW — Beneficio Facebook','Cada Aliada puede realizar una publicación comercial por semana en el grupo de Facebook, de acuerdo con las reglas de convivencia y moderación.','marketing','beneficio'),
 ('sw006','DirectorioSW — Beneficio WhatsApp','El canal comercial de WhatsApp permite como máximo un mensaje de venta diario por integrante, sujeto a sus normas internas.','marketing','beneficio'),
 ('sw007','DirectorioSW — Perfil público','Cada Aliada aprobada tiene un perfil público con información de su negocio, descripción, categoría y datos de contacto que decida compartir.','producto','beneficio'),
 ('sw008','DirectorioSW — Membresía','Las Aliadas Fundadoras reciben tres meses de membresía gratuita después de la activación y, posteriormente, la tarifa prevista es de 20000 pesos colombianos al mes.','finanzas','precio')]
OWN=[{'id':i,'title':t,'content':c,'metadata':{'department':d,'type':ty}} for i,t,c,d,ty in OWN]
Path('data/company_documents.json').write_text(json.dumps(OWN,ensure_ascii=False,indent=2),encoding='utf-8')
def build_store(items,chunk_size=400,overlap=40,collection='company'):
    splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=overlap)
    chunks=[]
    for item in items:
        for j,txt in enumerate(splitter.split_text(item['content'])):
            chunks.append(Document(page_content=txt,metadata={'title':item['title'],'source':item['id'],'chunk_index':j,**item['metadata']}))
    cli=QdrantClient(':memory:')
    cli.create_collection(collection_name=collection,vectors_config=models.VectorParams(size=EMBEDDING_DIM,distance=models.Distance.COSINE))
    store=QdrantVectorStore(client=cli,collection_name=collection,embedding=embeddings)
    store.add_documents(chunks)
    return store,chunks
store_demo,chunks_demo=build_store(DEMO,collection='demo')
store_company,chunks_company=build_store(OWN,chunk_size=400,overlap=40,collection='company')
print('✅ A1:',len(chunks_demo),'chunks; A2:',len(chunks_company),'chunks')

✅ A1: 4 chunks; A2: 8 chunks


In [8]:
PROMPT_BEFORE='Eres un asistente empresarial. Responde solo usando el contexto. Si no contiene la respuesta, indica que no tienes esa información. Responde en español y de forma concisa.'
PROMPT_AFTER='Eres un asistente empresarial. Responde SOLO usando el contexto. Cita el título de cada documento entre corchetes [Título]. Si el contexto no alcanza, responde EXACTAMENTE: No encuentro esa información en los documentos. Responde en español en máximo 3 oraciones.'
def ask(store,question,threshold=0.30,k=3,strict=True):
    scores=store.similarity_search_with_score(question,k=k)
    found=[(doc,float(score)) for doc,score in scores if score>=threshold]
    if not found: return 'No encuentro esa información en los documentos.',scores,found
    context='\n\n'.join(f'[{d.metadata["title"]}] {d.page_content}' for d,_ in found)
    if llm:
        try:
            prompt=ChatPromptTemplate.from_messages([('system',PROMPT_AFTER if strict else PROMPT_BEFORE),('human','Contexto:\n{context}\n\nPregunta: {question}')])
            answer=(prompt | llm | StrOutputParser()).invoke({'context':context,'question':question})
            return answer,scores,found
        except Exception as exc:
            print('⚠️ Error de generación:',str(exc)[:120])
    # Fallback EXTRACTIVO, nunca pretender que el LLM produjo esta respuesta
    evidence=' '.join(f'[{d.metadata["title"]}] {d.page_content}' for d,_ in found[:2])
    return '[RESPUESTA EXTRACTIVA, SIN LLM] '+evidence,scores,found
Q_A1=[
 '¿Cuál es el precio del plan Enterprise de CloudSync Pro?',
 '¿Cuántos días pueden trabajar remotamente los empleados?',
 '¿Qué pasa con los reembolsos mayores a $100?',
 '¿Qué ocurre durante la primera semana de onboarding?',
 '¿En qué ciudad está la oficina central de CloudSync Pro?']
Q_A2=[
 '¿Cuánto cuesta postularse al directorio?',
 '¿Cuántas publicaciones comerciales puede hacer una aliada en Facebook?',
 '¿Cuáles son los requisitos de ingreso?',
 '¿Qué incluye el perfil público?',
 '¿Cuántos meses de membresía gratuita tienen las fundadoras?',
 '¿Cuál es la dirección física de la oficina?',
 '¿Quién ganó el sorteo de 2027?']
def run_company(name,store,questions):
    rows=[]
    for q in questions:
        ans,scores,found=ask(store,q)
        rows.append({'Pregunta':q,'Respuesta':ans,'Fuentes':', '.join(dict.fromkeys(d.metadata['title'] for d,_ in found)) or 'Ninguna','Top 3 scores':', '.join(f'{s:.3f}' for _,s in scores)})
    df=pd.DataFrame(rows);section(name,table_md(df));display(df);return df
results_a1=run_company('A1 · Corpus de demostración',store_demo,Q_A1)
results_a2=run_company('A2 · Corpus propio (8 documentos)',store_company,Q_A2)
section('A2 · Justificación del chunking','Se eligió chunk_size=400 y overlap=40 porque los documentos propios son muy cortos: cada política normalmente ocupa un solo fragmento y se evita mezclar políticas distintas. Si crecieran a múltiples párrafos, el overlap conservaría parte del contexto entre límites.')
comparisons=[]
for label,q in [('Fácil',Q_A2[0]),('Ambigua','¿Qué ocurre si una postulante no cumple los requisitos?'),('Fuera de contexto',Q_A2[5])]:
    before,_,_=ask(store_company,q,strict=False)
    after,_,_=ask(store_company,q,strict=True)
    comparisons.append({'Tipo':label,'Pregunta':q,'Antes':before,'Después':after})
section('A3 · Comparación del prompt','**Antes:** '+PROMPT_BEFORE+'\n\n**Después:** '+PROMPT_AFTER+'\n\n'+table_md(pd.DataFrame(comparisons))+'\n\n**Para completar:** describe cuál instrucción funcionó mejor y cuál ignoró el modelo al revisar las respuestas. Si se usó respuesta extractiva, no es válido evaluar obediencia de un LLM.')
section('A4 · Scores y umbral', table_md(results_a2[['Pregunta','Top 3 scores','Fuentes']])+'\n\n**Umbral inicial propuesto:** 0.30 (provisional). Contrasta los scores de preguntas relevantes e irrelevantes, porque umbral óptimo depende del embedding y del corpus. El método de scores consulta sin filtrar antes de aplicar el umbral.')
print('✅ A1–A4 completados; quedan interpretaciones personales y capturas de interfaz')

,Pregunta,Respuesta,Fuentes,Top 3 scores
0,¿Cuál es el precio del plan Enterprise de Clou...,"[RESPUESTA EXTRACTIVA, SIN LLM] [CloudSync Pro...",CloudSync Pro — Precios,"0.707, 0.041, 0.040"
1,¿Cuántos días pueden trabajar remotamente los ...,"[RESPUESTA EXTRACTIVA, SIN LLM] [Política de t...",Política de trabajo remoto,"0.636, 0.099, 0.091"
2,¿Qué pasa con los reembolsos mayores a $100?,"[RESPUESTA EXTRACTIVA, SIN LLM] [Política de r...",Política de reembolsos,"0.612, 0.034, 0.015"
3,¿Qué ocurre durante la primera semana de onboa...,"[RESPUESTA EXTRACTIVA, SIN LLM] [Onboarding] D...",Onboarding,"0.597, 0.115, 0.052"
4,¿En qué ciudad está la oficina central de Clou...,"[RESPUESTA EXTRACTIVA, SIN LLM] [CloudSync Pro...",CloudSync Pro — Precios,"0.380, 0.058, 0.036"


,Pregunta,Respuesta,Fuentes,Top 3 scores
0,¿Cuánto cuesta postularse al directorio?,"[RESPUESTA EXTRACTIVA, SIN LLM] [DirectorioSW ...",DirectorioSW — Postulación,"0.310, 0.233, 0.195"
1,¿Cuántas publicaciones comerciales puede hacer...,"[RESPUESTA EXTRACTIVA, SIN LLM] [DirectorioSW ...","DirectorioSW — Beneficio Facebook, DirectorioS...","0.634, 0.357, 0.302"
2,¿Cuáles son los requisitos de ingreso?,No encuentro esa información en los documentos.,Ninguna,"0.139, 0.103, 0.100"
3,¿Qué incluye el perfil público?,"[RESPUESTA EXTRACTIVA, SIN LLM] [DirectorioSW ...",DirectorioSW — Perfil público,"0.435, 0.049, 0.040"
4,¿Cuántos meses de membresía gratuita tienen la...,"[RESPUESTA EXTRACTIVA, SIN LLM] [DirectorioSW ...",DirectorioSW — Membresía,"0.618, 0.295, 0.290"
5,¿Cuál es la dirección física de la oficina?,No encuentro esa información en los documentos.,Ninguna,"0.059, 0.047, 0.022"
6,¿Quién ganó el sorteo de 2027?,No encuentro esa información en los documentos.,Ninguna,"0.048, 0.034, 0.024"


✅ A1–A4 completados; quedan interpretaciones personales y capturas de interfaz


## Gradio empresa


In [9]:
import gradio as gr
def gradio_company(question,corpus):
    store=store_demo if corpus=='Demostración A1' else store_company
    ans,scores,found=ask(store,question,threshold=0.3,strict=True)
    scoretext='\n'.join(f'- **{d.metadata.get("title")}** — coseno {s:.3f}'+(' (bajo umbral)' if s<0.3 else '') for d,s in scores)
    return ans,scoretext or 'Sin fuentes'
with gr.Blocks() as demo_ui:
    gr.Markdown('# RAG · Workshop SI3003 | Fuentes y scores')
    corpus=gr.Dropdown(['Demostración A1','Propio A2'],value='Propio A2',label='Corpus')
    question=gr.Textbox(label='Pregunta',value='¿Cuánto cuesta postularse al directorio?')
    askbtn=gr.Button('Preguntar')
    answer=gr.Markdown(label='Respuesta'); sources=gr.Markdown(label='Fuentes')
    askbtn.click(gradio_company,[question,corpus],[answer,sources])
try:
    demo_ui.queue().launch(share=True,prevent_thread_lock=True,show_error=True)
except Exception as exc: print('Gradio:',str(exc)[:300])

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0bc01a7fc8c3c3165.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## B1-B5: Papers


In [10]:
import requests
from pypdf import PdfReader
PAPERS={'attention_paper.pdf':('Attention Is All You Need','https://arxiv.org/pdf/1706.03762'), 'rag_paper.pdf':('Retrieval-Augmented Generation','https://arxiv.org/pdf/2005.11401')}
Path('papers').mkdir(exist_ok=True)
PAPER_TEXT={}
for name,(title,url) in PAPERS.items():
    path=Path('papers')/name
    if not path.exists():
        resp=requests.get(url,timeout=90,headers={'User-Agent':'Mozilla/5.0'});resp.raise_for_status()
        if not resp.content.startswith(b'%PDF'): raise ValueError(f'{url} no devolvió un PDF')
        path.write_bytes(resp.content)
    reader=PdfReader(str(path))
    txt='\n'.join(page.extract_text() or '' for page in reader.pages)
    PAPER_TEXT[name]=txt
    print('✅',name,len(reader.pages),'páginas',len(txt),'caracteres')
attention_text=PAPER_TEXT['attention_paper.pdf'];rag_text=PAPER_TEXT['rag_paper.pdf']
def build_paper_store(size=900,overlap=None):
    overlap=round(size*0.1333) if overlap is None else overlap
    splitter=RecursiveCharacterTextSplitter(chunk_size=size,chunk_overlap=overlap,separators=['\n\n','\n','.',' '])
    docs=[]
    for name,(title,_) in PAPERS.items():
        for i,txt in enumerate(splitter.split_text(PAPER_TEXT[name])):
            if len(txt.strip())>=30:docs.append(Document(page_content=txt,metadata={'title':title,'source':name,'chunk_index':i}))
    cli=QdrantClient(':memory:'); cname='papers_'+str(size)
    cli.create_collection(collection_name=cname,vectors_config=models.VectorParams(size=EMBEDDING_DIM,distance=models.Distance.COSINE))
    vs=QdrantVectorStore(client=cli,collection_name=cname,embedding=embeddings)
    # inserción por lotes para no saturar NVIDIA
    for i in range(0,len(docs),32):vs.add_documents(docs[i:i+32])
    return vs,len(docs)
store_900,n900=build_paper_store(900,120)
print('✅',n900,'chunks indexados en Qdrant')

✅ attention_paper.pdf 15 páginas 39599 caracteres
✅ rag_paper.pdf 19 páginas 69062 caracteres
✅ 143 chunks indexados en Qdrant


In [11]:
# Preguntas centrales del quiz. Compara manualmente la respuesta con el ground truth del notebook 03 original.
QUIZ=[
 ('Q1','What BLEU score did the Transformer big model achieve on WMT 2014 English-to-German?','28.4 BLEU','attention_paper.pdf'),
 ('Q2','How many attention heads are used in the Transformer base model?','8 heads','attention_paper.pdf'),
 ('Q3','What scaling factor is used for scaled dot-product attention?','1/sqrt(d_k)','attention_paper.pdf'),
 ('Q4','What are RAG-Sequence and RAG-Token?','RAG-Sequence uses one retrieved document for the whole generated sequence; RAG-Token can condition each token on a different document.','rag_paper.pdf'),
 ('Q5','What two main components form a RAG model?','A neural retriever and a sequence-to-sequence generator.','rag_paper.pdf'),
 ('Bonus','How does RAG retrieve documents, and why is approximate nearest neighbor search important for both RAG and large vector indexes?','RAG relies on retrieval over a dense vector index; ANN improves scalability, while the Transformer supplies attention-based generation machinery.','rag_paper.pdf')]
def paper_ask(q,store=store_900,k=5,threshold=0.25):return ask(store,q,k=k,threshold=threshold)
def evaluate_quiz(store,k=5,threshold=0.25):
    rows=[]
    for id,q,gt,expected in QUIZ:
        ans,scores,found=paper_ask(q,store,k,threshold)
        papers_found=list(dict.fromkeys(d.metadata['source'] for d,_ in found))
        rows.append({'ID':id,'Pregunta':q,'Ground truth':gt,'Respuesta RAG':ans,'Papers recuperados':', '.join(papers_found),'Paper esperado recuperado':expected in papers_found,'Correcta (revisar)':'pendiente'})
    return pd.DataFrame(rows)
quiz_df=evaluate_quiz(store_900)
section('B1 · Quiz y ground truth',table_md(quiz_df))
display(quiz_df[['ID','Ground truth','Respuesta RAG','Papers recuperados']])
print('BONUS · papers recuperados:',quiz_df.iloc[-1]['Papers recuperados'])

,ID,Ground truth,Respuesta RAG,Papers recuperados
0,Q1,28.4 BLEU,"[RESPUESTA EXTRACTIVA, SIN LLM] [Attention Is ...",attention_paper.pdf
1,Q2,8 heads,"[RESPUESTA EXTRACTIVA, SIN LLM] [Attention Is ...",attention_paper.pdf
2,Q3,1/sqrt(d_k),"[RESPUESTA EXTRACTIVA, SIN LLM] [Attention Is ...",attention_paper.pdf
3,Q4,RAG-Sequence uses one retrieved document for t...,"[RESPUESTA EXTRACTIVA, SIN LLM] [Retrieval-Aug...",rag_paper.pdf
4,Q5,A neural retriever and a sequence-to-sequence ...,"[RESPUESTA EXTRACTIVA, SIN LLM] [Retrieval-Aug...",rag_paper.pdf
5,Bonus,RAG relies on retrieval over a dense vector in...,"[RESPUESTA EXTRACTIVA, SIN LLM] [Retrieval-Aug...",rag_paper.pdf


BONUS · papers recuperados: rag_paper.pdf


In [12]:
# B2: se reconstruye solo para 400 y 1500; el índice de 900 se reutiliza
store_400,n400=build_paper_store(400)
store_1500,n1500=build_paper_store(1500)
STORES={400:(store_400,n400),900:(store_900,n900),1500:(store_1500,n1500)}
# Proxy verificable de recuperación: ¿llega un fragmento del paper esperado?
# NO lo equipares con exactitud factual de la respuesta generada.
def hit_source(store,k,threshold=0.25):
    hits=0
    for _,q,_,source in QUIZ[:5]:
        _,_,found=paper_ask(q,store,k,threshold)
        hits+=any(d.metadata['source']==source for d,_ in found)
    return hits
experiment_rows=[]
for size in [400,900,1500]:
    st,nch=STORES[size]
    experiment_rows.append({'Experimento':'chunk_size','Valor':size,'N chunks':nch,'Hits fuente / 5':hit_source(st,5,.25),'k':5,'Umbral':.25})
for k in [3,5,8]:experiment_rows.append({'Experimento':'k','Valor':k,'N chunks':n900,'Hits fuente / 5':hit_source(store_900,k,.25),'k':k,'Umbral':.25})
for threshold in [.15,.25,.40]:experiment_rows.append({'Experimento':'threshold','Valor':threshold,'N chunks':n900,'Hits fuente / 5':hit_source(store_900,5,threshold),'k':5,'Umbral':threshold})
experiments=pd.DataFrame(experiment_rows)
section('B2 · Experimentos',table_md(experiments)+'\n\nLa columna «Hits fuente / 5» indica recuperación del paper esperado y NO necesariamente acierto factual de la respuesta. Evalúa manualmente las cinco respuestas generadas por configuración antes de informar aciertos reales. Compara cambios de cobertura, costo de indexación y posibles fragmentos irrelevantes.');display(experiments)
experiments.to_csv('experimentos_B2.csv',index=False)

,Experimento,Valor,N chunks,Hits fuente / 5,k,Umbral
0,chunk_size,400.00,308,5,5,0.25
1,chunk_size,900.00,143,5,5,0.25
2,chunk_size,1500.00,85,5,5,0.25
3,k,3.00,143,5,3,0.25
4,k,5.00,143,5,5,0.25
5,k,8.00,143,5,8,0.25
6,threshold,0.15,143,5,5,0.15
7,threshold,0.25,143,5,5,0.25
8,threshold,0.40,143,5,5,0.40


In [ ]:
TRAPS=[
 'What was the exact BLEU score of GPT-5 on the WMT 2014 test set, according to these two papers?',
 'Which NVIDIA GPU model did the authors of the 2020 RAG paper use to benchmark a 2025 language model?',
 'How many downloads did the Transformer paper receive in October 2026?']
trap_rows=[]
for q in TRAPS:
    ans,_,found=paper_ask(q)
    trap_rows.append({'Pregunta':q,'Respuesta':ans,'Clasificación (verificar)':'pendiente: reconoce / inventa / mezcla','Fuentes':', '.join(dict.fromkeys(d.metadata['source'] for d,_ in found))})
section('B3 · Preguntas trampa',table_md(pd.DataFrame(trap_rows))+'\n\nSi detectas una invención, refuerza el system prompt con una prohibición explícita de dar fechas o cifras ausentes y vuelve a ejecutar las preguntas.');display(pd.DataFrame(trap_rows))
LANGUAGE=[(QUIZ[0][1],'¿Qué puntuación BLEU logró Transformer big en WMT 2014 de inglés a alemán?'),(QUIZ[2][1],'¿Cuál es el factor de escalamiento de la atención de producto punto escalada?'),(QUIZ[3][1],'¿Cuál es la diferencia entre RAG-Sequence y RAG-Token?')]
lang_rows=[]
for en,es in LANGUAGE:
    a_en,_,f_en=paper_ask(en);a_es,_,f_es=paper_ask(es)
    ids=lambda fs:set((d.metadata['source'],d.metadata['chunk_index']) for d,_ in fs)
    lang_rows.append({'EN':en,'ES':es,'Chunks iguales':ids(f_en)==ids(f_es),'Chunks comunes':len(ids(f_en)&ids(f_es)),'Respuesta EN':a_en,'Respuesta ES':a_es})
section('B4 · Español vs inglés',table_md(pd.DataFrame(lang_rows))+'\n\nHipótesis: un embedding multilingüe alinea preguntas de diferentes idiomas en espacios cercanos; se comprueba midiendo la coincidencia de chunks y la similitud de las respuestas con una evaluación humana.');display(pd.DataFrame(lang_rows)[['EN','ES','Chunks iguales','Chunks comunes']])
NEW_QUIZ=[
 ('What is the positional encoding based on in the Transformer?','Sine and cosine functions of different frequencies.','attention_paper.pdf','Section 3.5'),
 ('What is the role of the document retriever in RAG?','It retrieves relevant passages from an external knowledge index to condition the generator.','rag_paper.pdf','Section 2'),
 ('How can a Transformer decoder be complemented with RAG document retrieval?','The Transformer provides attention-based generation, and RAG conditions generation on retrieved passages from an external index.','rag_paper.pdf','Attention Section 3 / RAG Section 2')]
new_rows=[]
for q,gt,source,sec in NEW_QUIZ:
    ans,_,found=paper_ask(q)
    new_rows.append({'Pregunta':q,'Ground truth':gt,'Sección':sec,'Respuesta':ans,'Fuentes':', '.join(dict.fromkeys(d.metadata['source'] for d,_ in found))})
section('B5 · Tres preguntas nuevas',table_md(pd.DataFrame(new_rows)));display(pd.DataFrame(new_rows))

## Retos: C3, C4 y C5


In [ ]:
# C4: 10 criterios específicos de recuperación; revisar las claves exactas en los chunks PDF.
EVAL=[
 ('How many attention heads are used in the Transformer base model?','attention_paper.pdf','8'),
 ('What is the BLEU score of Transformer big on English-to-German?','attention_paper.pdf','28.4'),
 ('Which dataset is used for English to German translation?','attention_paper.pdf','WMT'),
 ('What optimizer is used to train the Transformer?','attention_paper.pdf','Adam'),
 ('What is the Transformer based on?','attention_paper.pdf','attention'),
 ('What is RAG-Token?','rag_paper.pdf','RAG-Token'),
 ('What is RAG-Sequence?','rag_paper.pdf','RAG-Sequence'),
 ('Which retriever is used in RAG?','rag_paper.pdf','DPR'),
 ('What does RAG combine?','rag_paper.pdf','retriev'),
 ('How are retrieved documents represented in RAG?','rag_paper.pdf','document')]
eval_df=pd.DataFrame(EVAL,columns=['pregunta','fuente_esperada','palabra_clave']);eval_df.to_csv('eval/preguntas.csv',index=False)
def hit_at_k(store,rows,k):
    hits=0
    for row in rows:
        matches=store.similarity_search(row['pregunta'],k=k)
        hits+=any(row['palabra_clave'].casefold() in d.page_content.casefold() and d.metadata['source']==row['fuente_esperada'] for d in matches)
    return hits/len(rows)
metric_rows=[]
for size,(st,_) in STORES.items():
    for k in [3,5]:metric_rows.append({'chunk_size':size,'k':k,'hit@k':hit_at_k(st,eval_df.to_dict('records'),k)})
metric_df=pd.DataFrame(metric_rows)
section('C4 · Hit@k',table_md(metric_df)+'\n\n**Limitación 1:** una coincidencia literal de palabra clave no garantiza que el fragmento explique correctamente el hecho. **Limitación 2:** paráfrasis, saltos de línea y errores de extracción de PDF producen falsos negativos aunque exista evidencia semántica.');display(metric_df)
# C5: MMR vs similitud
mmr_rows=[]
for label,q in [('Dato puntual',QUIZ[0][1]),('Explicación RAG',QUIZ[3][1]),('Dos papers',QUIZ[-1][1])]:
    sim=store_900.as_retriever(search_kwargs={'k':5}).invoke(q)
    mmr=store_900.as_retriever(search_type='mmr',search_kwargs={'k':5,'fetch_k':20,'lambda_mult':0.5}).invoke(q)
    fmt=lambda ds:', '.join(f'{d.metadata["source"]}#{d.metadata["chunk_index"]}' for d in ds)
    mmr_rows.append({'Pregunta':label,'Similitud':fmt(sim),'MMR':fmt(mmr),'Fuentes distintas (sim)':len(set(d.metadata['source'] for d in sim)),'Fuentes distintas (MMR)':len(set(d.metadata['source'] for d in mmr))})
mmr_df=pd.DataFrame(mmr_rows)
section('C5 · Similitud y MMR',table_md(mmr_df)+'\n\nMMR puede beneficiar preguntas abiertas o que necesitan ambas fuentes si aumenta diversidad; puede perjudicar preguntas puntuales cuando reemplaza fragmentos muy relevantes por otros menos pertinentes.');display(mmr_df)

In [ ]:
# C3: app con filtro Qdrant por metadata.source y scores
from qdrant_client.models import Filter,FieldCondition,MatchValue
OPTIONS={'Todos':None,'Attention Is All You Need':'attention_paper.pdf','Retrieval-Augmented Generation':'rag_paper.pdf'}
def papers_ui(question,paper):
    source=OPTIONS[paper]
    filt=Filter(must=[FieldCondition(key='metadata.source',match=MatchValue(value=source))]) if source else None
    scored=store_900.similarity_search_with_score(question,k=5,filter=filt)
    found=[(d,float(s)) for d,s in scored if s>=0.25]
    if not found: answer='No encuentro esa información en los documentos.'
    else:
        context='\n\n'.join(f'[{d.metadata["title"]} | {d.metadata["source"]}] {d.page_content}' for d,_ in found)
        if llm:
            try:
                p=ChatPromptTemplate.from_messages([('system','Use only evidence in context. Cite paper titles. If missing evidence, say you do not know. Respond in the question language.'),('human','Context: {context}\nQuestion: {question}')])
                answer=(p|llm|StrOutputParser()).invoke({'context':context,'question':question})
            except Exception as exc: answer='Error LLM: '+str(exc)[:200]
        else: answer='[EXTRACTIVO SIN LLM] '+context[:1200]
    sources='\n'.join(f'- {d.metadata["title"]} | {d.metadata["source"]} | score {s:.3f}' for d,s in scored)
    return answer,sources
with gr.Blocks() as paper_app:
    gr.Markdown('# RAG Papers · Filtro por documento')
    q=gr.Textbox(label='Pregunta',value=QUIZ[3][1])
    paper=gr.Dropdown(list(OPTIONS),value='Todos',label='Paper')
    btn=gr.Button('Consultar')
    ans=gr.Markdown();refs=gr.Markdown()
    btn.click(papers_ui,[q,paper],[ans,refs])
try:
    paper_app.queue().launch(share=True,prevent_thread_lock=True,show_error=True)
except Exception as exc: print('Gradio papers:',str(exc)[:300])
compare=[]
for opt in ['Todos','Retrieval-Augmented Generation','Attention Is All You Need']:
    a,s=papers_ui(QUIZ[3][1],opt);compare.append({'Filtro':opt,'Respuesta':a,'Fuentes y scores':s})
section('C3 · Comparación del filtro por paper',table_md(pd.DataFrame(compare)));display(pd.DataFrame(compare)[['Filtro','Respuesta']])

## Reflexión y archivos


In [ ]:
reflection="""### 1. Contexto insuficiente
Las preguntas fuera de contexto permiten distinguir una negativa fundamentada de una respuesta inventada. En la tabla B3 se observa la salida real, que hay que clasificar como reconocimiento, invención o mezcla. Para reducir alucinaciones se debe imponer un mensaje fijo y verificar el origen de cada afirmación.

### 2. Pregunta bonus
La pregunta bonus combina recuperación semántica y generación con apoyo de papers. La tabla B1 muestra qué archivos aparecieron en los fragmentos recuperados y permite verificar si están los dos. La ausencia de un paper señala una limitación del retriever, del vocabulario de la consulta o de la selección de k.

### 3. Parámetros
La tabla B2 compara chunk_size, k y score_threshold usando el mismo conjunto de preguntas. Fragmentos pequeños pueden localizar datos puntuales con precisión, pero perder contexto; fragmentos grandes aumentan el contexto y el costo por inserción. Aumentar k mejora cobertura potencial, mientras que subir el umbral puede excluir evidencia relevante.

### 4. Modelos separados
El embedding traduce textos y preguntas a representaciones comparables para recuperación. El generador convierte el contexto recuperado en respuestas interpretables para el usuario. Separarlos permite actualizar el generador sin recalcular todos los embeddings, siempre que se conserve el mismo espacio vectorial para consultas y documentos.

### 5. Producción
En producción reemplazaría Qdrant en memoria por almacenamiento persistente, respaldos y políticas de acceso. Agregaría una evaluación de recuperación con ground truth, pruebas de alucinaciones, métricas de latencia y trazas de los chunks usados. También limitaría los costos de llamadas a modelos, gestionaría secretos fuera del repositorio y vigilaría la calidad tras cada cambio de corpus.
"""
section('R1 · Reflexión',reflection)
# Reemplazar los datos de auditoría según el fallback real
section('Nota metodológica','Modelo de embeddings: '+ ('NVIDIA '+EMBEDDING_MODEL if isinstance(embeddings,NVIDIAEmbeddings) else 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (fallback local)')+'\n\nModelo generativo: '+(GENERATION_MODEL if llm else 'SIN LLM, respuestas extractivas de respaldo')+'\n\nLos puntos marcados «pendiente» requieren evaluación humana; el score de similitud no es una nota automática de exactitud.')
Path('RESPUESTAS.md').write_text('\n'.join(RESPUESTAS),encoding='utf-8')
readme="""# SI3003 Workshop RAG

Cuaderno integral de experimentos y evidencias. Completar nombre, código estudiantil, análisis personal y capturas antes de entregar.

- [RESPUESTAS.md](RESPUESTAS.md)
- [data/company_documents.json](data/company_documents.json)
- [eval/preguntas.csv](eval/preguntas.csv)
- [experimentos_B2.csv](experimentos_B2.csv)

**No subir claves API.**
"""
Path('README.md').write_text(readme,encoding='utf-8')
import zipfile
with zipfile.ZipFile('entrega_workshop_rag.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in ['RESPUESTAS.md','README.md','data/company_documents.json','eval/preguntas.csv','experimentos_B2.csv']:z.write(f)
print('✅ Archivos generados: RESPUESTAS.md, README.md, corpus, CSV de evaluación y experimentos, ZIP.')
print('⚠️ Faltan capturas de Gradio y completar análisis experimental y etiquetas de corrección.')
try:
    from google.colab import files
    files.download('entrega_workshop_rag.zip')
except ImportError: pass